In [1]:
import torch
import torch.nn as nn
import gensim
from datasets import load_dataset
from collections import Counter

ds = load_dataset("Ayon128/Banglish-English")
print(ds['train'][0])

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/479k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4000 [00:00<?, ? examples/s]

{'Banglish': 'Amar ei guitar ta cai.', 'English': 'I want this guitar.'}


In [3]:
train_data = ds['train']
test_data = ds['test']

english_sentences_train = []
banglish_sentences_train = []
english_sentences_test = []
banglish_sentences_test = []

for item in train_data:
    english_sentences_train.append(item['English'])
    banglish_sentences_train.append(item['Banglish'])

for item in test_data:
    english_sentences_test.append(item['English'])
    banglish_sentences_test.append(item['Banglish'])
    
english_sentences_test[:5]


['Fogging off in the afternoon',
 '"An astrologer from Howrah gave one of his clients this coral. But it\'s not real coral. Ordinary people can tell."',
 "he should've been fired months ago.",
 'While nutrition is undoubtedly a cornerstone of physical health, it is just one piece of the puzzle.',
 "I don't like coffee."]

In [5]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
def preprocess_string(s):
    # Remove all non-word characters (everything except numbers and letters)
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)
    return s

def preprocess_sentence(sentences, vocab):
    processed_sentences = []
    for s in sentences:
        indices = []
        preprocess_string(s)
        indices.append(vocab.get(SOS_TOKEN, vocab[UNK_TOKEN]))
        for word in s.split():
             indices.append(vocab.get(word.lower(), vocab[UNK_TOKEN]))
        indices.append(vocab.get(EOS_TOKEN, vocab[UNK_TOKEN]))
        processed_sentences.append(indices)
    return processed_sentences
    
def vocab_build(data):
    tokens = []
    tokens.append(PAD_TOKEN)
    tokens.append(SOS_TOKEN)
    tokens.append(EOS_TOKEN)
    tokens.append(UNK_TOKEN)
    for sentence in data:
        for word in sentence.split():
            tokens.append(word.lower())
    word_freq = Counter(tokens)
    idx = 0
    vocab = {}
    for word, _ in word_freq.items():
        vocab[word] = idx
        idx += 1    
    return vocab
    
s = ['jhow are you', 'you are who']
word_freq = vocab_build(s)
ds = Counter(word_freq)
print(word_freq)
        

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'jhow': 4, 'are': 5, 'you': 6, 'who': 7}


In [6]:


class Encoder(nn.Module):
    def __init__(self, emb_size, hidden_dim, num_layers, vocab_size, batch_size):
        super(Encoder, self).__init__()
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding (vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size= emb_size,hidden_size = hidden_dim,num_layers=num_layers,batch_first=True)

    def forward(self, data):
        h_0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(data.device)
        c_0 = torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(data.device)
        embedded_data = self.embedding(data)
        out, (hidden, cell) = self.lstm(embedded_data, (h_0, c_0))
        return out, hidden, cell

In [14]:
class Decoder(nn.Module):
    def __init__( self, emb_size,hidden_dim, num_layers, vocab_size, batch_size):
        super(Decoder, self).__init__()
        self.emb_size = emb_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size=emb_size, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, data, hidden, cell):
        embedded_data = self.embedding(data) 
        out, (hidden, cell) = self.lstm(embedded_data, (hidden, cell))
        output = self.fc(out.squeeze(1))  # (batch_size, vocab_size)
        return output, hidden, cell


In [20]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder,device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, vocab_size).to(self.device)
        hidden, cell = self.encoder(source)
        input = target[:, 0]
        for t in range(1, target_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output
            input = target[:, t] if torch.rand(1).item() < teacher_force_ratio else output.argmax(1)
        return outputs
        


In [21]:
EPOCHS = 5
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EMB_SIZE = 100
HIDDEN_DIM = 100
NUM_LAYERS = 1


banglish_vocab = vocab_build(banglish_sentences_train)
english_vocab = vocab_build(english_sentences_train)
VOCAB_SIZE_BANGLISH = len(banglish_vocab)
VOCAB_SIZE_ENGLISH = len(english_vocab)

encoder = Encoder(EMB_SIZE, HIDDEN_DIM, NUM_LAYERS, VOCAB_SIZE_BANGLISH, BATCH_SIZE)
decoder = Decoder(EMB_SIZE, HIDDEN_DIM, NUM_LAYERS, VOCAB_SIZE_ENGLISH, BATCH_SIZE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Seq2Seq2(encoder, decoder, device)
criterion = nn.CrossEntropyLoss(ignore_index=english_vocab['<PAD>'])
optimizer = torch.optim.Adam(seq2seq.parameters(), lr=LEARNING_RATE)

def train(model, data_loader, optimizer, criterion, device, epochs):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        epoch_correct = 0  # To count the number of correct predictions
        epoch_total = 0    # Total number of tokens for accuracy calculation
        
        for source, target in data_loader:
            source, target = source.to(device), target.to(device)
            optimizer.zero_grad()
            
            # Forward pass through the model
            output = model(source, target)
    
            # Ignore <SOS> token and reshape for loss calculation
            output = output[:, 1:].reshape(-1, output.shape[2])  # Ignore <SOS>, shape: (batch * seq_length, voc_size)
            target = target[:, 1:].reshape(-1)  # shape: (batch_size * seq_length)

            # Calculate loss
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            # Update epoch loss
            epoch_loss += loss.item()

            # Calculate accuracy
            _, predicted = output.max(1)  # Get the index of the max log-probability
            epoch_correct += (predicted == target).sum().item()  # Count correct predictions
            epoch_total += target.size(0)  # Total number of tokens processed

        # Print loss and accuracy for the current epoch
        epoch_accuracy = epoch_correct / epoch_total * 100
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(data_loader):.4f}, Accuracy: {epoch_accuracy:.2f}%")

    return epoch_loss / len(data_loader)

    
def collate_fn(batch, pad_idx):
    """
    Custom function to collate a batch of variable-length sequences
    and pad them to the same length.
    
    batch: List of tuples (input_sentence, target_sentence)
    pad_idx: Index of the padding token
    """
    source_sentences, target_sentences = zip(*batch)  # Unpack input-output pairs

    # Pad source and target sequences
    source_padded = pad_sequence(source_sentences, batch_first=True, padding_value=pad_idx)
    target_padded = pad_sequence(target_sentences, batch_first=True, padding_value=pad_idx)

    return source_padded, target_padded

pad_idx = 0  # Define padding index (usually 0)
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=lambda batch: collate_fn(batch, pad_idx))    
train_model(model, data_loader, optimizer, criterion)

TypeError: super(type, obj): obj must be an instance or subtype of type

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for source, target in test_loader:
            source, target = source.to(device), target.to(device)
            output = model(source, target, teacher_force_ratio=0)
            output = output.argmax(2)
            correct += (output == target).sum().item()
            total += target.numel()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

evaluate(model, test_loader)

In [29]:
import torch

output = torch.rand((2, 4, 5)) 
print(output)

tensor([[[0.1427, 0.9295, 0.7896, 0.4420, 0.6370],
         [0.2933, 0.3392, 0.3487, 0.9090, 0.9616],
         [0.5797, 0.3344, 0.8440, 0.0173, 0.7322],
         [0.8402, 0.1017, 0.5765, 0.0665, 0.5990]],

        [[0.6092, 0.0282, 0.2569, 0.4264, 0.2895],
         [0.9230, 0.6952, 0.1515, 0.7481, 0.6703],
         [0.5363, 0.6297, 0.7720, 0.3640, 0.3893],
         [0.4250, 0.4688, 0.8565, 0.1098, 0.5524]]])


In [45]:
cv = output
output_dropped = output[:, 1:, :]
print('----', output_dropped.reshape(-1))
# o = output_dropped.reshape(-1)
# print(o)

cv = cv[:, 1:].reshape(-1, cv.shape[2])
print(cv)

---- tensor([0.2933, 0.3392, 0.3487, 0.9090, 0.9616, 0.5797, 0.3344, 0.8440, 0.0173,
        0.7322, 0.8402, 0.1017, 0.5765, 0.0665, 0.5990, 0.9230, 0.6952, 0.1515,
        0.7481, 0.6703, 0.5363, 0.6297, 0.7720, 0.3640, 0.3893, 0.4250, 0.4688,
        0.8565, 0.1098, 0.5524])
tensor([[0.2933, 0.3392, 0.3487, 0.9090, 0.9616],
        [0.5797, 0.3344, 0.8440, 0.0173, 0.7322],
        [0.8402, 0.1017, 0.5765, 0.0665, 0.5990],
        [0.9230, 0.6952, 0.1515, 0.7481, 0.6703],
        [0.5363, 0.6297, 0.7720, 0.3640, 0.3893],
        [0.4250, 0.4688, 0.8565, 0.1098, 0.5524]])


In [7]:
target = torch.randint(0, 5, (2, 4)) 
print(target)

tensor([[4, 4, 2, 0],
        [2, 2, 3, 3]])


In [ ]:
import torch

# Simulated model output (logits) before softmax
# Shape: (batch_size, sequence_length, vocab_size)
output = torch.rand((2, 4, 5))  # Batch of 2, 4 time steps, 5 possible classes (vocab size)

# Target tensor (actual labels), each token is a class index
# Shape: (batch_size, sequence_length)
target = torch.randint(0, 5, (2, 4))  # Batch of 2, 4 time steps, class indices in range [0,4]

print("Original Output Shape:", output.shape)  # (2, 4, 5)
print("Original Target Shape:", target.shape)  # (2, 4)

# Ignore the first token (<SOS>), so we remove the first column (axis=1)
output = output[:, 1:]  # Now shape is (2, 3, 5)
target = target[:, 1:]  # Now shape is (2, 3)

print("\nAfter Removing <SOS>:")
print("Output Shape:", output.shape)  # (2, 3, 5)
print("Target Shape:", target.shape)  # (2, 3)

# Reshape for loss computation
output = output.reshape(-1, output.shape[2])  # (2*3, 5) = (6, 5)
target = target.reshape(-1)  # (2*3,) = (6,)